# Préparer une carte mel pour Conv2d : valeurs, échelle et axes

Passer d’une onde à une entrée (extraits,canaux,fréquences,temps) sans transformation cachée.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#preparer-mel-pour-conv).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
try:
    import torchaudio
except (ImportError, OSError) as error:
    raise RuntimeError("TorchAudio manque ou ne correspond pas à PyTorch. Installer torch==2.10.0 et torchaudio==2.10.0 ensemble, puis redémarrer.") from error
print("TorchAudio",torchaudio.__version__)

TorchAudio 2.10.0+cpu


In [3]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Nous changeons de données pour étudier les convolutions : deux sinusoïdes synthétiques remplacent les trois tableaux de descripteurs. Elles isolent les formes sans dépendre d’un fichier à télécharger.

Chaque onde dure une seconde à 16 000 échantillons par seconde. Leur amplitude choisie est 0,2 ; leurs fréquences choisies sont 220 et 440 Hz. Ce ne sont pas des enregistrements de voix.

| Paramètre | Valeur | Conséquence |
| --- | --- | --- |
| n_fft et win_length | 400 échantillons | 201 coefficients de fréquences positives, zéro compris |
| hop_length | 160 échantillons | Une nouvelle trame tous les 10 ms |
| center | False | Pas de prolongement centré au début et à la fin |
| n_mels | 64 | 64 bandes mel pour chaque trame |

Avec 16 000 valeurs et une fenêtre de 400, on obtient floor((16000−400)/160)+1 = 98 trames.

Le programme calcule d’abord la STFT complexe, puis une puissance regroupée en bandes mel. Hann décrit la fenêtre. Slaney fixe ici les conventions de filtres ; les détails restent dans le module Signal.

### Deux choix de préparation à comprendre avant leur utilisation

Nous prenons un logarithme avec un plancher de 10⁻¹⁰, pour éviter log(0). Une puissance numérique de 0,01 donne −20 dB par rapport à la référence numérique 1.

Diviser ensuite ces nombres par 100 donne −0,2 dans cette miniature. C’est une mise à l’échelle choisie pour le réseau, pas une conversion en probabilités ni une mesure de sonie.

Nous gardons 96 trames sur les 98. Cette découpe enlève les deux dernières trames temporelles, pour rendre les tailles divisibles par quatre avant deux réductions par deux.

Enfin, unsqueeze(1) ajoute un canal de caractéristiques. Cela ne transforme pas le son mono en stéréo : cet axe appartient au réseau.

## Démonstration 1 · Lire les états de la préparation

Le programme autonome fabrique lui-même ses deux ondes.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [4]:
import torch
import torchaudio


### Fixer les conditions du petit calcul

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [5]:
torch.manual_seed(17)


### Calculer l’état suivant

arange(sr)/sr donne les instants en secondes. Chaque sinusoïde contient sr valeurs ; stack crée l’axe des deux extraits.

In [6]:
sr = 16000
temps = torch.arange(sr) / sr
onde_220 = 0.2 * torch.sin(2 * torch.pi * 220 * temps)
onde_440 = 0.2 * torch.sin(2 * torch.pi * 440 * temps)


In [7]:
observer('sr', sr)
observer('temps', temps)
observer('onde_220', onde_220)
observer('onde_440', onde_440)

sr = 16000
temps forme (16000,) type torch.float32 requires_grad False
onde_220 forme (16000,) type torch.float32 requires_grad False
onde_440 forme (16000,) type torch.float32 requires_grad False


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [8]:
ondes = torch.stack([onde_220, onde_440])


In [9]:
observer('ondes', ondes)

ondes forme (2, 16000) type torch.float32 requires_grad False


### Calculer l’état suivant

torch.stft produit des coefficients complexes. MelSpectrogram choisit ici power=2 : les bandes regroupent des puissances, pas des phases.

In [10]:
fenetre = torch.hann_window(400)
stft = torch.stft(ondes, n_fft=400, hop_length=160, window=fenetre,
                  center=False, return_complex=True)
transformation = torchaudio.transforms.MelSpectrogram(
    sample_rate=sr, n_fft=400, win_length=400, hop_length=160,
    n_mels=64, f_min=0., f_max=8000., center=False,
    mel_scale="slaney", norm="slaney", power=2.)


In [11]:
observer('fenetre', fenetre)
observer('stft', stft)
observer('transformation', transformation)

fenetre forme (400,) type torch.float32 requires_grad False
stft forme (2, 201, 98) type torch.complex64 requires_grad False


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [12]:
mel_brut = transformation(ondes)


In [13]:
observer('mel_brut', mel_brut)

mel_brut forme (2, 64, 98) type torch.float32 requires_grad False


### Calculer l’état suivant

La division par la somme de Hann au carré suit notre convention spectrale. Le plancher est appliqué avant log10 ; le facteur 10 correspond à un rapport de puissances.

In [14]:
mel = mel_brut / fenetre.sum().square()
mel_positif = mel.clamp_min(1e-10)
mel_db = 10 * mel_positif.log10()


In [15]:
observer('mel', mel)
observer('mel_positif', mel_positif)
observer('mel_db', mel_db)

mel forme (2, 64, 98) type torch.float32 requires_grad False
mel_positif forme (2, 64, 98) type torch.float32 requires_grad False
mel_db forme (2, 64, 98) type torch.float32 requires_grad False


### Calculer l’état suivant

Le facteur 100 ne change pas les axes. :96 sélectionne les 96 premières positions du dernier axe, celui du temps.

Le premier : garde les deux extraits ; le second garde toutes les bandes ; :96 limite seulement le temps.

In [16]:
mel_reduit = mel_db / 100
mel_court = mel_reduit[:, :, :96]


In [17]:
observer('mel_reduit', mel_reduit)
observer('mel_court', mel_court)

mel_reduit forme (2, 64, 98) type torch.float32 requires_grad False
mel_court forme (2, 64, 96) type torch.float32 requires_grad False


### Calculer l’état suivant

L’axe 1 est inséré entre extraits et bandes. Les valeurs restent les mêmes ; seule leur organisation en quatre axes change.

In [18]:
entree = mel_court.unsqueeze(1)


In [19]:
observer('entree', entree)

entree forme (2, 1, 64, 96) type torch.float32 requires_grad False


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [20]:
print("ondes :", tuple(ondes.shape))
print("STFT :", tuple(stft.shape))
print("mel :", tuple(mel.shape))
print("carte coupee :", tuple(mel_court.shape))


ondes : (2, 16000)
STFT : (2, 201, 98)
mel : (2, 64, 98)
carte coupee : (2, 64, 96)


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [21]:
print("entree Conv2d :", tuple(entree.shape))


entree Conv2d : (2, 1, 64, 96)


### Calculer l’état suivant

Nous choisissons une puissance de 0,01. Le logarithme donne −20 dB ; la division par 100 donne ensuite −0,2, sans changer le sens des axes.

In [22]:
puissance_exemple = torch.tensor(0.01)
db_exemple = 10 * puissance_exemple.log10()


In [23]:
observer('puissance_exemple', puissance_exemple)
observer('db_exemple', db_exemple)

puissance_exemple forme () type torch.float32 requires_grad False
  valeurs : 0.009999999776482582
db_exemple forme () type torch.float32 requires_grad False
  valeurs : -20.0


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [24]:
print("echelle :", db_exemple.item(), "dB ->", round((db_exemple / 100).item(), 3))

echelle : -20.0 dB -> -0.2


L’architecture recevra une carte logarithmique recadrée, pas l’onde ni la carte mel brute.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Combien de bins donne la partie non redondante d’une FFT réelle de taille 400 ?

In [25]:
exo_n_fft = 400
exo_bins = ...  # Remplacer les points par votre réponse.
if exo_bins is Ellipsis or exo_bins is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_bins == torch.fft.rfft(torch.zeros(exo_n_fft)).numel()
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

Les fréquences zéro et de Nyquist sont présentes pour cette taille paire : N//2+1.

</details>

### Exercice 2

Un mel a la forme (2 extraits,64 bandes,96 trames). Ajoutez le canal mono attendu par Conv2d. Quelle forme ?

In [26]:
exo_mel = torch.zeros(2,64,96)
exo_forme = ...  # Remplacer les points par votre réponse.
if exo_forme is Ellipsis or exo_forme is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_forme) == tuple(exo_mel.unsqueeze(1).shape)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Le canal s’insère après les extraits. Les bandes restent un axe spatial de la représentation.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_n_fft = 400
exo_bins = 201
if exo_bins is Ellipsis or exo_bins is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_bins == torch.fft.rfft(torch.zeros(exo_n_fft)).numel()
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_mel = torch.zeros(2,64,96)
exo_forme = (2,1,64,96)
if exo_forme is Ellipsis or exo_forme is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_forme) == tuple(exo_mel.unsqueeze(1).shape)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [27]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()